# Web Scraping Moderno con BeautifulSoup4 y Requests

## 🎯 Objetivos de Aprendizaje
- Comprender los fundamentos del protocolo HTTP y la arquitectura del DOM HTML.
- Extraer datos estructurados utilizando selectores CSS y métodos de búsqueda en `BeautifulSoup4`.
- Implementar buenas prácticas de scraping: User-Agents, cabeceras, límites de tasa y `robots.txt`.
- Conocer alternativas modernas para páginas dinámicas con renderizado JavaScript (Selenium, Playwright, Scrapy).
- Construir un pipeline modular de extracción, limpieza y almacenamiento con Pandas en Python 3.12+.

## 🌉 Puente Pedagógico: Leyendo la Web como Datos

### ¿Por qué hacer Web Scraping?
La inmensa mayoría de la información en internet no cuenta con una API oficial. El web scraping permite transformar páginas web pensadas para visualización humana en tablas estructuradas listas para análisis.

### Analogía
Imagina que visitas un supermercado:
- **Navegador web**: Eres tú recorriendo los pasillos y viendo las etiquetas de precios.
- **Web Scraper**: Un asistente automatizado que lee selectivamente las etiquetas de un estante específico y las anota en una libreta ordenada.

### Diagrama del Flujo de Scraping
```
  [ Servidor Web ] <--- (1) HTTP GET Request (con User-Agent) --- [ Python Requests ]
         |                                                                |
         +------------ (2) HTML crudo (String) ---------------------------+
                                                                          |
                                                                          v
                                                             [ BeautifulSoup (DOM Parser) ]
                                                                          |
                                           (3) Selectores CSS / find() ---+
                                                                          |
                                                                          v
                                                             [ Registros Limpios / Tablas ]
                                                                          |
                                                                          v
                                                             [ DataFrame Pandas / CSV ]
```

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# Simulación pedagógica de HTML para garantizar reproducibilidad sin depender de la red
html_doc = """
<!DOCTYPE html>
<html>
<head><title>Librería de Data Science</title></head>
<body>
    <h1>Catálogo de Libros Recomendados</h1>
    <div class="contenedor">
        <div class="libro" data-id="101">
            <h3 class="titulo">Python Crash Course</h3>
            <p class="autor">Eric Matthes</p>
            <span class="precio">$29.99</span>
            <span class="stock">Disponible</span>
        </div>
        <div class="libro" data-id="102">
            <h3 class="titulo">Fluent Python</h3>
            <p class="autor">Luciano Ramalho</p>
            <span class="precio">$45.50</span>
            <span class="stock">Agotado</span>
        </div>
        <div class="libro" data-id="103">
            <h3 class="titulo">Designing Data-Intensive Applications</h3>
            <p class="autor">Martin Kleppmann</p>
            <span class="precio">$38.00</span>
            <span class="stock">Disponible</span>
        </div>
    </div>
</body>
</html>
"""

soup = BeautifulSoup(html_doc, "html.parser")
print("Título de la página:", soup.title.string)

## 1. Extracción Estructurada con Selectores CSS

Podemos navegar la jerarquía del DOM utilizando `.select()` con sintaxis de selectores CSS estándar.

In [ ]:
libros_encontrados = soup.select(".libro")
datos = []

for item in libros_encontrados:
    titulo = item.select_one(".titulo").get_text(strip=True)
    autor = item.select_one(".autor").get_text(strip=True)
    precio_raw = item.select_one(".precio").get_text(strip=True)
    precio = float(precio_raw.replace("$", ""))
    stock = item.select_one(".stock").get_text(strip=True)
    item_id = int(item["data-id"])
    
    datos.append({
        "id": item_id,
        "titulo": titulo,
        "autor": autor,
        "precio_usd": precio,
        "disponible": stock == "Disponible"
    })

df_libros = pd.DataFrame(datos)
display(df_libros)

## 2. Ética, Legalidad y Alternativas para la Web Moderna

### Buenas Prácticas y Ética
1. **Respetar `robots.txt`**: Revisa siempre `https://dominio.com/robots.txt` antes de iniciar scraping masivo.
2. **Rate Limiting**: Agrega pausas (`time.sleep(1)`) para no sobrecargar los servidores destino.
3. **Cabecera `User-Agent`**: Identifícate claramente para evitar bloqueos automatizados.

### Comparativa de Herramientas de Extracción

| Herramienta | Velocidad | Soporta JS dinámico | Complejidad | Caso de Uso Ideal |
|:---|:---|:---|:---|:---|
| **BeautifulSoup4 + Requests** | ⚡ Muy Alta | ❌ No | Muy Baja | Páginas estáticas HTML simples |
| **Scrapy** | ⚡⚡ Extrema (Asíncrono) | ⚠️ Con plugins | Media/Alta | Crawling masivo a escala industrial |
| **Playwright / Selenium** | 🐢 Moderada | ✅ Sí (Navegador real) | Media | Single Page Applications (React, Vue, Angular) |

## 📝 Ejercicios Prácticos

### Ejercicio 1 (Guiado): Función modular para extraer links
Crea una función que reciba código HTML y devuelva una lista de tuplas con el texto del enlace y su destino `href`.

In [ ]:
def extraer_enlaces(html: str) -> list[dict[str, str]]:
    s = BeautifulSoup(html, "html.parser")
    return [{"texto": a.get_text(strip=True), "href": a.get("href", "")} for a in s.find_all("a")]

html_prueba = '<nav><a href="/inicio">Inicio</a> | <a href="/contacto">Contacto</a></nav>'
print(extraer_enlaces(html_prueba))

### Ejercicio 2 (Independiente): Filtrado de Catálogo
Filtra el DataFrame `df_libros` para mostrar únicamente los libros con precio menor a $40 que además estén en stock.

In [ ]:
# Solución:
filtro = df_libros[(df_libros["precio_usd"] < 40) & (df_libros["disponible"])]
display(filtro)

## 📋 Resumen
- BeautifulSoup4 simplifica la navegación del árbol DOM permitiendo búsquedas por etiqueta, clase o selector CSS.
- Siempre sanitiza los datos numéricos y de texto antes de agregarlos al DataFrame.
- Ante páginas con renderizado dinámico por cliente (JS), recurre a herramientas de automatización de navegador como Playwright.